# Pixel3DMM V4 — Hair App Multi-Photo Full Pipeline

공식 Pixel3DMM 전체 실행 흐름에 Hair App용 사진별 no-roll crop을 통합한 노트북이다.

- audited upstream commit: fcd1fa973c7715b02a8948dfc679dff53cf85924
- full flow: 환경 → FLAME assets → FaceBoxes per-image crop → PIPNet 98 landmarks → FaRL segmentation → normal/UV → FLAME tracking → mesh 저장
- crop contract: official FaceBoxes, margin 1.42, 512×512, no roll, per-image affine metadata
- removed: independent photos의 shared static bbox, RetinaFace sparse 5-point roll(v1~v3)
- license: Pixel3DMM CC BY-NC 4.0, FLAME assets 별도 연구 라이선스

> 위에서 아래로 실행한다. condacolab.install() 뒤 런타임 재시작은 정상이다.
> preprocessing 시각 gate에서 한 번 멈춘다. crop·PIPNet 98점·segmentation을 확인한 뒤 승인해야 normal/UV와 3D tracking이 실행된다.
> private 사진, landmark, mask, mesh, notebook output은 git에 올리지 않는다.


## 0. GPU 확인 및 CUDA arch 자동 기록

In [ ]:
!nvidia-smi
import pathlib, torch
assert torch.cuda.is_available(), 'GPU runtime이 아님: 런타임 유형을 GPU로 변경하세요.'
GPU_NAME = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
TORCH_ARCH = f'{cc[0]}.{cc[1]}+PTX'
pathlib.Path('/content/p3dmm_torch_arch.txt').write_text(TORCH_ARCH)
print('GPU:', GPU_NAME, '| compute capability:', cc, '| TORCH_CUDA_ARCH_LIST:', TORCH_ARCH)
assert cc in {(8, 0), (9, 0)}, '이 노트북은 A100 또는 H100 기준. 다른 GPU면 arch/dependency를 재검증하세요.'

## 1. Conda 설치 — 이 셀 뒤 런타임 자동 재시작

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

## 2. 공식 저장소 clone 및 audited commit 고정

In [ ]:
import condacolab; condacolab.check()
import os, pathlib
%cd /content
if not os.path.exists('/content/pixel3dmm/.git'):
    !git clone https://github.com/SimonGiebenhain/pixel3dmm.git
%cd /content/pixel3dmm
!git fetch origin
!git checkout fcd1fa973c7715b02a8948dfc679dff53cf85924
!git rev-parse HEAD

## 3. p3dmm 환경 생성 및 CUDA extension 빌드

공식 environment의 Torch 2.7/cu118 조합을 명시적으로 고정한다. 빌드는 10~25분 걸릴 수 있다.

In [ ]:
%%bash
set -euo pipefail
if ! conda env list | awk '{print $1}' | grep -qx p3dmm; then
  conda create -n p3dmm python=3.9 -y
fi
conda run -n p3dmm pip install \
  torch==2.7.0+cu118 torchvision==0.22.0+cu118 torchaudio==2.7.0+cu118 \
  --index-url https://download.pytorch.org/whl/cu118
conda install -n p3dmm -y \
  nvidia/label/cuda-11.8.0::cuda-nvcc nvidia/label/cuda-11.8.0::cuda-cccl \
  nvidia/label/cuda-11.8.0::cuda-cudart nvidia/label/cuda-11.8.0::cuda-cudart-dev \
  nvidia/label/cuda-11.8.0::libcusparse nvidia/label/cuda-11.8.0::libcusparse-dev \
  nvidia/label/cuda-11.8.0::libcublas nvidia/label/cuda-11.8.0::libcublas-dev \
  nvidia/label/cuda-11.8.0::libcurand nvidia/label/cuda-11.8.0::libcurand-dev \
  nvidia/label/cuda-11.8.0::libcusolver nvidia/label/cuda-11.8.0::libcusolver-dev
conda run -n p3dmm nvcc --version

In [ ]:
%%bash
set -euo pipefail
ENVDIR=/usr/local/envs/p3dmm
ARCH=$(cat /content/p3dmm_torch_arch.txt)
export CUDA_HOME=$ENVDIR
export TORCH_CUDA_ARCH_LIST=$ARCH
conda run -n p3dmm pip install ninja fvcore iopath
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST=$ARCH \
  conda run -n p3dmm pip install --no-build-isolation 'git+https://github.com/facebookresearch/pytorch3d.git@75ebeeaea0908c5527e7b1e305fbc7681382db47'
CUDA_HOME=$ENVDIR TORCH_CUDA_ARCH_LIST=$ARCH \
  conda run -n p3dmm pip install --no-build-isolation 'git+https://github.com/NVlabs/nvdiffrast.git@253ac4fcea7de5f396371124af597e6cc957bfae'
cd /content/pixel3dmm
conda run -n p3dmm pip install -r requirements.txt
conda run -n p3dmm pip install -e .
conda run -n p3dmm python - <<'PY'
import torch, pytorch3d, nvdiffrast.torch as dr, pixel3dmm
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.get_device_name(0))
print('core imports: PASS')
PY

## 4. 전처리 의존성 설치 — 과거 오류 수정 포함

- SSH clone 대신 HTTPS
- FaceBoxes 전에 Cython 설치
- checkpoint를 공식 코드가 읽는 `pretrained_weights/`에 저장
- `ignore_mica=True`인데도 원본 tracker가 MICA 파일을 강제로 읽는 부분을 zero prior로 패치
- 전처리에서 불필요한 MICA 실행을 건너뜀

In [ ]:
%%bash
set -euo pipefail
PRE=/content/pixel3dmm/src/pixel3dmm/preprocessing
conda run -n p3dmm pip install -q Cython gdown
rm -rf "$PRE/facer" "$PRE/PIPNet"
cd "$PRE"
git clone https://github.com/FacePerceiver/facer.git
cd facer
git checkout ddd35c76ff840174b8a5403ad1c1255e37b8782b
cp ../replacement_code/farl.py facer/face_parsing/farl.py
cp ../replacement_code/facer_transform.py facer/transform.py
conda run -n p3dmm pip install -e .
cd "$PRE"
git clone https://github.com/jhb86253817/PIPNet.git
cd PIPNet
git checkout b9eab58816437403a34aa5bc3adeafe5081fd36b
cd ..
cd PIPNet/FaceBoxesV2/utils
conda run -n p3dmm sh make.sh
cd "$PRE/PIPNet"
mkdir -p snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10
conda run -n p3dmm gdown 1nVkaSbxy3NeqblwMTGvLg4nF49cI_99C \
  -O snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth
mkdir -p /content/pixel3dmm/pretrained_weights "$PRE/MICA/data"
cd /content/pixel3dmm/pretrained_weights
conda run -n p3dmm gdown 1SDV_8_qWTe__rX_8e4Fi-BE3aES0YzJY -O uv.ckpt
conda run -n p3dmm gdown 1KYYlpN-KGrYMVcAOT22NkVQC0UAfycMD -O normals.ckpt
test $(stat -c%s uv.ckpt) -gt 1000000
test $(stat -c%s normals.ckpt) -gt 1000000
test $(stat -c%s "$PRE/PIPNet/snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth") -gt 1000000
# FaRL 617MB weight는 on-demand 단일 요청이 자주 끊기므로 retry/resume로 미리 받는다.
FARL_DIR=/root/.cache/torch/hub/checkpoints
FARL_NAME=face_parsing.farl.celebm.main_ema_181500_jit.pt
FARL_URL=https://github.com/FacePerceiver/facer/releases/download/models-v1/$FARL_NAME
mkdir -p "$FARL_DIR"
if [ ! -s "$FARL_DIR/$FARL_NAME" ]; then
  touch "$FARL_DIR/$FARL_NAME.part"
  curl -L --fail --retry 20 --retry-all-errors --retry-delay 2 --connect-timeout 30 \
    --continue-at - --output "$FARL_DIR/$FARL_NAME.part" "$FARL_URL"
  mv "$FARL_DIR/$FARL_NAME.part" "$FARL_DIR/$FARL_NAME"
fi
conda run -n p3dmm python -c "import torch; p='$FARL_DIR/$FARL_NAME'; torch.jit.load(p,map_location='cpu'); print('FaRL JIT weight: PASS', p)"
echo 'preprocessing dependencies/checkpoints: PASS'

In [ ]:
from pathlib import Path
import re

# 1) facer 최신 torch index dtype 오류
farl = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/facer/facer/face_parsing/farl.py')
text = farl.read_text()
text = text.replace("images[data['image_ids']]", "images[data['image_ids'].long()]")
farl.write_text(text)
assert "images[data['image_ids'].long()]" in farl.read_text()

# 2) ignore_mica=True 경로에서는 전처리 MICA 실행 자체를 생략
pre = Path('/content/pixel3dmm/scripts/run_preprocessing.py')
text = pre.read_text()
mica_call = "    os.system(f'cd {env_paths.CODE_BASE}/src/pixel3dmm/preprocessing/MICA ; python demo.py -video_name {vid_name} -a {env_paths.PREPROCESSED_DATA}/{vid_name}/arcface/')"
assert mica_call in text or 'HAIR_APP_SKIP_MICA' in text
text = text.replace(mica_call, "    print('HAIR_APP_SKIP_MICA: ignore_mica=True tracking will use a zero shape prior')")
pre.write_text(text)

# 3) upstream tracker는 ignore_mica=True여도 mica/identity.npy를 먼저 읽으므로 zero prior로 우회
tracker = Path('/content/pixel3dmm/src/pixel3dmm/tracking/tracker.py')
text = tracker.read_text()
pattern = re.compile(r"        mica_folder = f'\{DATA_FOLDER\}/mica'.*?            mica_shape = np.mean\(mica_shapes, axis=0\)\n", re.S)
replacement = """        # HAIR_APP_ZERO_MICA_PRIOR
        if self.config.ignore_mica:
            mica_shape = np.zeros(self.config.num_shape_params, dtype=np.float32)
        else:
            mica_folder = f'{DATA_FOLDER}/mica'
            mica_files = os.listdir(mica_folder)
            mica_shapes = []
            for mica_file in mica_files:
                one_shape = np.load(f'{mica_folder}/{mica_file}/identity.npy')
                mica_shapes.append(np.squeeze(one_shape))
            mica_shapes = np.stack(mica_shapes, axis=0)
            mica_shape = mica_shapes[0, :] if self.config.early_exit else np.mean(mica_shapes, axis=0)
"""
if 'HAIR_APP_ZERO_MICA_PRIOR' not in text:
    text, changed = pattern.subn(replacement, text)
    assert changed == 1, f'tracker patch count={changed}'
tracker.write_text(text)

# 4) PyTorch 2.6+ weights_only=True 기본값과 official Pixel3DMM Lightning checkpoint 호환
inference = Path('/content/pixel3dmm/scripts/network_inference.py')
text = inference.read_text()
old_load = 'model = p3dmm_system.load_from_checkpoint(model_checkpoint, strict=False)'
new_load = 'model = p3dmm_system.load_from_checkpoint(model_checkpoint, strict=False, weights_only=False)'
assert old_load in text or new_load in text
text = text.replace(old_load, new_load)
inference.write_text(text)
assert new_load in inference.read_text()
print('facer + skip-MICA + tracker zero-prior + trusted checkpoint patches: PASS')

## 5. Google Drive 마운트 및 FLAME2020 + FLAME2023 수동 설치

1. <https://flame.is.tue.mpg.de/> 로그인 및 약관 동의
2. `FLAME2020.zip`, `FLAME2023.zip`을 직접 다운로드
3. Drive `MyDrive/hair_app/models/`에 zip 그대로 업로드

자동 wget은 로그인 HTML을 zip처럼 저장할 수 있어 사용하지 않는다. Pixel3DMM은 FLAME2023 mesh를 쓸 때도 FLAME2020의 generic model, landmark, masks를 읽으므로 **두 zip 모두 필수**다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pickle, shutil, zipfile
import numpy as np

models = Path('/content/drive/MyDrive/hair_app/models')
models.mkdir(parents=True, exist_ok=True)
assets = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/MICA/data')
dst20 = assets / 'FLAME2020'
dst23 = assets / 'FLAME2023'
unpack = Path('/content/flame_assets_v4')
shutil.rmtree(unpack, ignore_errors=True)
unpack.mkdir(parents=True)

archives = sorted(models.rglob('*.zip'))
assert archives, f'FLAME 관련 zip이 없음: {models}'
for archive in archives:
    if archive.stat().st_size < 100_000 or not zipfile.is_zipfile(archive):
        print('skip invalid/small zip:', archive)
        continue
    target = unpack / archive.stem
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(target)
    print('extracted:', archive.name)

search_roots = [models, unpack]
def find_exact(name, prefer=None):
    found = []
    for root in search_roots:
        found.extend(root.rglob(name))
    found = [p for p in found if p.is_file() and p.stat().st_size > 0]
    if prefer:
        preferred = [p for p in found if prefer.lower() in str(p).lower()]
        if preferred:
            found = preferred
    assert found, f'{name}을 {models} 또는 압축 해제 폴더에서 찾지 못함'
    return sorted(found, key=lambda p: len(str(p)))[0]

shutil.rmtree(dst20, ignore_errors=True)
shutil.rmtree(dst23, ignore_errors=True)
(dst20 / 'FLAME_masks').mkdir(parents=True)
dst23.mkdir(parents=True)

generic = find_exact('generic_model.pkl', prefer='2020')
flame23 = find_exact('flame2023_no_jaw.pkl', prefer='2023')
masks = find_exact('FLAME_masks.pkl')
shutil.copy2(generic, dst20 / 'generic_model.pkl')
shutil.copy2(flame23, dst23 / 'flame2023_no_jaw.pkl')
shutil.copy2(masks, dst20 / 'FLAME_masks' / 'FLAME_masks.pkl')

existing_embedding = []
for root in search_roots:
    existing_embedding.extend(root.rglob('landmark_embedding.npy'))
existing_embedding = [
    p for p in existing_embedding
    if p.is_file() and p.stat().st_size > 0 and 'MICA/data/FLAME2020' not in str(p)
]
if existing_embedding:
    embedding_src = sorted(existing_embedding, key=lambda p: len(str(p)))[0]
    shutil.copy2(embedding_src, dst20 / 'landmark_embedding.npy')
    print('using existing landmark_embedding.npy:', embedding_src)
else:
    # FLAME2020/2023/Vertex Masks ZIP에는 이 파일이 없으므로 DECA가 사용하는
    # FLAME landmark embedding을 pinned commit에서 받고 SHA-256까지 검증한다.
    import hashlib, urllib.request
    embedding_url = (
        'https://raw.githubusercontent.com/yfeng95/DECA/'
        'a11554ae2a2b0f3998cf1fa94dd4db03babb34a2/data/landmark_embedding.npy'
    )
    embedding_dst = dst20 / 'landmark_embedding.npy'
    urllib.request.urlretrieve(embedding_url, embedding_dst)
    embedding_sha256 = hashlib.sha256(embedding_dst.read_bytes()).hexdigest()
    expected_sha256 = '8095348eeafce5a02f6bd8765146307f9567a3f03b316d788a2e47336d667954'
    assert embedding_sha256 == expected_sha256, (embedding_sha256, expected_sha256)
    embedding = np.load(embedding_dst, allow_pickle=True, encoding='latin1')[()]
    required_embedding_keys = {
        'static_lmk_faces_idx', 'static_lmk_bary_coords',
        'dynamic_lmk_faces_idx', 'dynamic_lmk_bary_coords',
    }
    assert required_embedding_keys.issubset(embedding.keys()), embedding.keys()
    print('downloaded pinned DECA landmark_embedding.npy: PASS')

required = [
    dst20 / 'generic_model.pkl',
    dst20 / 'landmark_embedding.npy',
    dst20 / 'FLAME_masks' / 'FLAME_masks.pkl',
    dst23 / 'flame2023_no_jaw.pkl',
]
for path in required:
    assert path.exists() and path.stat().st_size > 0, f'필수 FLAME asset 없음: {path}'
    print(path.name, '->', path, 'PASS')
print('ALL FLAME ASSETS: PASS')


## 6. Pixel3DMM 경로 설정 및 private 입력 확인

사진을 Drive `MyDrive/hair_app/inputs/`에 넣는다. 정면/좌우 3·4/좌우 profile/hairline 노출을 포함해 5장 이상 권장한다.

In [ ]:
import os, pathlib
cfg = pathlib.Path.home() / '.config' / 'pixel3dmm'
cfg.mkdir(parents=True, exist_ok=True)
(cfg / '.env').write_text(
    'PIXEL3DMM_CODE_BASE="/content/pixel3dmm"\n'
    'PIXEL3DMM_PREPROCESSED_DATA="/content/p3dmm_preprocessed"\n'
    'PIXEL3DMM_TRACKING_OUTPUT="/content/p3dmm_tracking"\n'
)
INPUT_PATH = '/content/drive/MyDrive/hair_app/inputs'
VID_NAME = os.path.basename(INPUT_PATH.rstrip('/'))
os.makedirs(INPUT_PATH, exist_ok=True)
imgs = sorted(f for f in os.listdir(INPUT_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print('input:', INPUT_PATH, '| VID_NAME:', VID_NAME, '| images:', len(imgs), imgs)
assert len(imgs) >= 2, '최소 2장 필요. 품질 비교에는 5장 이상 권장.'

## 7. V4 사진별 no-roll crop 및 전처리 검증

공식 FaceBoxes와 1.42 margin은 유지한다. 다만 동영상용 static_crop처럼 서로 다른 사진의 bbox를 평균내지 않고 사진마다 별도 bbox를 사용한다.

이 단계에서는 roll·yaw·pitch를 변경하지 않는다. 저장되는 512×512 crop이 PIPNet, FaRL, Pixel3DMM normal/UV, tracker가 공유하는 공통 image 좌표계다.


In [ ]:
import shutil, subprocess, textwrap
from pathlib import Path

root = Path('/content/p3dmm_preprocessed') / VID_NAME
assert str(root).startswith('/content/p3dmm_preprocessed/'), root
shutil.rmtree(root, ignore_errors=True)

helper = Path('/content/per_image_no_roll_crop_v4.py')
helper.write_text(textwrap.dedent(r'''
import argparse, json, math, sys
from pathlib import Path
import cv2, numpy as np, torch
from PIL import Image, ImageOps
from pixel3dmm import env_paths
faceboxes_dir = f'{env_paths.CODE_BASE}/src/pixel3dmm/preprocessing/PIPNet/FaceBoxesV2'
if faceboxes_dir not in sys.path:
    sys.path.insert(0, faceboxes_dir)
from faceboxes_detector import FaceBoxesDetector

SUFFIXES = {'.jpg', '.jpeg', '.png'}

def choose_primary(detections, width, height):
    faces = [d for d in detections if d[0] == 'face']
    if not faces:
        raise RuntimeError('FaceBoxes returned no face')
    areas = [max(float(d[4]), 0) * max(float(d[5]), 0) for d in faces]
    max_area = max(areas)
    half_diag = math.hypot(width, height) / 2
    ranks = []
    for index, (det, area) in enumerate(zip(faces, areas)):
        cx, cy = float(det[2])+float(det[4])/2, float(det[3])+float(det[5])/2
        center = max(0, 1-math.hypot(cx-width/2, cy-height/2)/half_diag)
        confidence = float(det[1])
        # official pipnet_utils.py처럼 FaceBoxes confidence를 우선한다.
        # area/center는 provenance와 향후 identity tie-break 연구용으로만 저장한다.
        score = confidence
        ranks.append({
            'detector_index': index, 'selection_score': score,
            'area_score': area/max(max_area, 1e-9),
            'confidence_score': confidence, 'center_score': center,
            'bbox_xywh': [float(v) for v in det[2:6]],
        })
    selected = max(range(len(ranks)), key=lambda i: ranks[i]['selection_score'])
    return faces[selected], selected, ranks

def square_box(det, width, height, margin):
    x, y, w, h = [float(v) for v in det[2:6]]
    cx, cy = x+w/2, y+h/2
    requested = max(w, h)*margin
    side = min(requested, float(width), float(height))
    side = max(2, int(round(side)))
    left, top = int(round(cx-side/2)), int(round(cy-side/2))
    left, top = min(max(left, 0), width-side), min(max(top, 0), height-side)
    warnings = ['source_too_tight_margin_reduced'] if side+1e-6 < requested else []
    return left, top, left+side, top+side, requested, warnings

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--input-dir', type=Path, required=True)
    p.add_argument('--output-root', type=Path, required=True)
    p.add_argument('--margin', type=float, default=1.42)
    args = p.parse_args()
    files = sorted(x for x in args.input_dir.iterdir() if x.is_file() and x.suffix.lower() in SUFFIXES)
    if not files:
        raise RuntimeError(f'no input images: {args.input_dir}')
    rgb, cropped, meta = args.output_root/'rgb', args.output_root/'cropped', args.output_root/'crop_meta'
    for directory in (rgb, cropped, meta):
        directory.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    weights = f'{env_paths.CODE_BASE}/src/pixel3dmm/preprocessing/PIPNet/FaceBoxesV2/weights/FaceBoxesV2.pth'
    detector = FaceBoxesDetector('FaceBoxes', weights, torch.cuda.is_available(), device)
    items = []
    for index, path in enumerate(files):
        with Image.open(path) as opened:
            source = ImageOps.exif_transpose(opened).convert('RGB')
        width, height = source.size
        bgr = cv2.cvtColor(np.asarray(source), cv2.COLOR_RGB2BGR)
        detections, _ = detector.detect(bgr, 0.6, 1)
        selected, selected_index, rankings = choose_primary(detections, width, height)
        left, top, right, bottom, requested, warnings = square_box(selected, width, height, args.margin)
        if float(selected[1]) < 0.75:
            warnings.append('low_faceboxes_confidence')
        if len(rankings) > 1:
            warnings.append('multiple_faces_detected')
        crop = source.crop((left, top, right, bottom)).resize((512, 512), Image.Resampling.LANCZOS)
        name = f'{index:05d}.jpg'
        crop.save(rgb/name, quality=95)
        crop.save(cropped/name, quality=95)
        side, scale = right-left, 512.0/(right-left)
        item = {
            'version': '0.4', 'engine': 'pixel3dmm_faceboxes_per_image_no_roll',
            'source_name': path.name, 'derived_name': name,
            'source_size': [width, height], 'output_size': [512, 512],
            'detector': 'official_faceboxes_v2', 'bbox_margin_requested': args.margin,
            'crop_box_ltrb': [left, top, right, bottom],
            'crop_side_requested': requested, 'crop_side_applied': side,
            'roll_applied': False,
            'source_to_crop': [[scale,0,-left*scale],[0,scale,-top*scale],[0,0,1]],
            'crop_to_source': [[1/scale,0,float(left)],[0,1/scale,float(top)],[0,0,1]],
            'selected_detector_index': selected_index,
            'candidate_rankings': rankings, 'warnings': warnings,
        }
        (meta/f'{index:05d}.json').write_text(json.dumps(item, indent=2, ensure_ascii=False))
        items.append(item)
        print(f"V4 CROP PASS {path.name} -> {name} score={float(selected[1]):.3f} warnings={warnings}")
    manifest = {
        'version':'0.4', 'engine':'pixel3dmm_faceboxes_per_image_no_roll',
        'count':len(items), 'margin':args.margin, 'roll_applied':False, 'items':items,
    }
    (meta/'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print('V4 PER-IMAGE NO-ROLL CROP COMPLETE:', len(items))

if __name__ == '__main__':
    main()
'''))
subprocess.run([
    'conda','run','--no-capture-output','-n','p3dmm','python',str(helper),
    '--input-dir',INPUT_PATH,'--output-root',str(root),'--margin','1.42',
], check=True)
print('V4 per-image FaceBoxes no-roll crop: PASS')


### 7.1 원본/crop 시각 gate

모든 crop에서 올바른 사용자가 선택됐고 눈·코·입·턱·필요한 이마/귀가 보이는지 확인한다. 얼굴이 기울어져 있어도 정상이다. 이 단계는 roll을 바로잡는 단계가 아니다.


In [ ]:
import json
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches

manifest = json.loads((root/'crop_meta'/'manifest.json').read_text())
fig, axes = plt.subplots(len(manifest['items']), 2, figsize=(12,4*len(manifest['items'])), squeeze=False)
for row, item in enumerate(manifest['items']):
    source = ImageOps.exif_transpose(Image.open(Path(INPUT_PATH)/item['source_name'])).convert('RGB')
    crop = Image.open(root/'cropped'/item['derived_name']).convert('RGB')
    axes[row,0].imshow(source)
    for candidate in item['candidate_rankings']:
        x,y,w,h = candidate['bbox_xywh']
        color = 'lime' if candidate['detector_index']==item['selected_detector_index'] else 'red'
        axes[row,0].add_patch(patches.Rectangle((x,y),w,h,fill=False,color=color,linewidth=2))
    axes[row,0].set_title(f"원본: {item['source_name']}\nwarnings={item['warnings']}")
    axes[row,1].imshow(crop)
    axes[row,1].set_title(f"V4 no-roll crop: {item['derived_name']}\n512x512, margin=1.42")
    axes[row,0].axis('off'); axes[row,1].axis('off')
plt.tight_layout(); plt.show()
print('CROP VISUAL GATE: 기울기는 그대로 두고 coverage와 주 피사체만 확인')


### 7.2 final crop에서 official PIPNet 98 landmark와 FaRL segmentation 실행

PIPNet은 final 512 crop 안에서 FaceBoxes로 temporary ROI를 다시 만들지만 persistent crop 파일을 바꾸지 않는다. 98개 landmark는 다시 final-crop 좌표로 환산된다. FaRL 재검출은 segmentation parser용 별도 단계다.


In [ ]:
import shutil, subprocess
from pathlib import Path

pipnet_utils = Path('/content/pixel3dmm/src/pixel3dmm/preprocessing/pipnet_utils.py')
text = pipnet_utils.read_text()
if 'HAIR_APP_V4_LANDMARK_THRESHOLD' not in text:
    old = "if detections[i][1] < 0.99:"
    assert old in text
    text = text.replace(old, "if detections[i][1] < 0.75:  # HAIR_APP_V4_LANDMARK_THRESHOLD")
crop_info_old = "if not os.path.exists(f'{image_dir}/../crop_ymin_ymax_xmin_xmax.npy'):"
if 'HAIR_APP_V4_DISABLE_CROP_METADATA_GUARD' not in text:
    assert crop_info_old in text
    text = text.replace(
        crop_info_old,
        "if (not disable_cropping) and not os.path.exists(f'{image_dir}/../crop_ymin_ymax_xmin_xmax.npy'):  # HAIR_APP_V4_DISABLE_CROP_METADATA_GUARD",
    )
pipnet_utils.write_text(text)
for name in ('PIPnet_landmarks','PIPnet_annotated_images','pipnet','seg_og','seg_non_crop_annotations'):
    shutil.rmtree(root/name, ignore_errors=True)
pip_helper = Path('/content/run_pipnet_on_v4_crop.py')
pip_helper.write_text("""
import sys
sys.path.insert(0, '/content/pixel3dmm/scripts')
from run_cropping import run
run(
    'experiments/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10.py',
    sys.argv[1], start_frame=-1, static_crop=True, max_bbox=True, disable_cropping=True,
)
""")
with open('/content/p3dmm_preprocess.log','w') as log:
    subprocess.run(
        ['conda','run','-n','p3dmm','python',str(pip_helper),str(root/'rgb')],
        stdout=log, stderr=subprocess.STDOUT, check=True,
    )
    subprocess.run(
        ['conda','run','--no-capture-output','-n','p3dmm','python','/content/pixel3dmm/scripts/run_facer_segmentation.py','--video_name',VID_NAME],
        stdout=log, stderr=subprocess.STDOUT, check=True,
    )
print(Path('/content/p3dmm_preprocess.log').read_text()[-12000:])


In [ ]:
from pathlib import Path
import json, matplotlib.pyplot as plt
from PIL import Image

manifest = json.loads((root/'crop_meta'/'manifest.json').read_text())
cropped = sorted((root/'cropped').glob('*.jpg'))
crop_meta = sorted(p for p in (root/'crop_meta').glob('*.json') if p.name!='manifest.json')
landmarks = sorted((root/'PIPnet_landmarks').glob('*.npy'))
annotated = sorted((root/'PIPnet_annotated_images').glob('*.jpg'))
seg = sorted((root/'seg_og').glob('*.png'))
print('input/crop/meta/landmark/annotated/seg:',len(imgs),len(cropped),len(crop_meta),len(landmarks),len(annotated),len(seg))
assert len(cropped)==len(imgs), 'crop 개수 불일치'
assert len(crop_meta)==len(cropped), 'crop metadata 누락'
assert len(landmarks)==len(cropped), 'PIPNet 98 landmark 누락: preprocess log 확인'
assert len(annotated)==len(cropped), 'PIPNet overlay 누락'
assert len(seg)==len(cropped), 'FaRL segmentation 누락: preprocess log 확인'
assert all(p.stat().st_size>0 for p in cropped+crop_meta+landmarks+annotated+seg)

fig, axes = plt.subplots(len(manifest['items']),3,figsize=(15,4*len(manifest['items'])),squeeze=False)
for row,item in enumerate(manifest['items']):
    name, stem = item['derived_name'], Path(item['derived_name']).stem
    axes[row,0].imshow(Image.open(root/'cropped'/name).convert('RGB')); axes[row,0].set_title(f'final crop {name}')
    axes[row,1].imshow(Image.open(root/'PIPnet_annotated_images'/name).convert('RGB')); axes[row,1].set_title('PIPNet 98 landmarks')
    seg_viz = root/'seg_non_crop_annotations'/f'color_{stem}.png'
    if seg_viz.exists():
        axes[row,2].imshow(Image.open(seg_viz).convert('RGB')); axes[row,2].set_title('FaRL segmentation')
    else:
        axes[row,2].imshow(Image.open(root/'seg_og'/f'{stem}.png'),cmap='tab20'); axes[row,2].set_title('FaRL labels')
    for axis in axes[row]: axis.axis('off')
plt.tight_layout(); plt.show()
PREPROCESSING_APPROVED = False
print('PREPROCESSING COMPLETE: PASS')
print('STOP: 결과를 확인한 뒤 PREPROCESSING_APPROVED=True로 바꾸세요.')


### 7.3 preprocessing 결과를 Google Drive에 보존

현재 runtime의 crop, PIPNet 98점, FaRL mask, 3열 overview를 private Drive run 폴더에 저장한다. 이 artifact는 biometric-sensitive data이므로 git에 넣지 않는다.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, shutil
import matplotlib.pyplot as plt
from PIL import Image

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
save_dir = Path('/content/drive/MyDrive/hair_app/runs') / f'pixel3dmm_v4_preprocessing_{VID_NAME}_{run_id}'
save_dir.mkdir(parents=True, exist_ok=False)

artifact_dirs = [
    'cropped', 'crop_meta', 'PIPnet_landmarks', 'PIPnet_annotated_images',
    'pipnet', 'seg_og', 'seg_non_crop_annotations',
]
copied = {}
for name in artifact_dirs:
    source = root / name
    if source.exists():
        shutil.copytree(source, save_dir / name)
        copied[name] = len([p for p in source.rglob('*') if p.is_file()])

manifest = json.loads((root/'crop_meta'/'manifest.json').read_text())
fig, axes = plt.subplots(len(manifest['items']), 3, figsize=(15, 4*len(manifest['items'])), squeeze=False)
for row, item in enumerate(manifest['items']):
    name, stem = item['derived_name'], Path(item['derived_name']).stem
    axes[row,0].imshow(Image.open(root/'cropped'/name).convert('RGB'))
    axes[row,0].set_title(f'final crop {name}')
    axes[row,1].imshow(Image.open(root/'PIPnet_annotated_images'/name).convert('RGB'))
    axes[row,1].set_title('PIPNet 98 landmarks')
    seg_viz = root/'seg_non_crop_annotations'/f'color_{stem}.png'
    if seg_viz.exists():
        axes[row,2].imshow(Image.open(seg_viz).convert('RGB'))
    else:
        axes[row,2].imshow(Image.open(root/'seg_og'/f'{stem}.png'), cmap='tab20')
    axes[row,2].set_title('FaRL segmentation')
    for axis in axes[row]: axis.axis('off')
plt.tight_layout()
overview = save_dir / 'preprocessing_overview.png'
fig.savefig(overview, dpi=160, bbox_inches='tight')
plt.close(fig)

summary = {
    'pipeline': 'pixel3dmm_colab_v4',
    'stage': 'preprocessing_visual_gate',
    'video_name': VID_NAME,
    'input_count': len(imgs),
    'artifact_file_counts': copied,
    'preprocessing_approved': bool(PREPROCESSING_APPROVED),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'privacy': 'private biometric artifacts; do not commit to git',
}
(save_dir/'preprocessing_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print('PREPROCESSING SAVED TO DRIVE:', save_dir)
print(json.dumps(summary, indent=2, ensure_ascii=False))


### 7.4 complete preprocessing reproducibility bundle

98-point 좌표를 NPY뿐 아니라 JSON/CSV와 512 crop pixel 좌표로도 내보내고, raw input, crop transform, segmentation 통계, 로그, SHA-256 manifest까지 같은 private Drive run에 보존한다.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv, hashlib, json, shutil, subprocess
import numpy as np
from PIL import Image

if 'save_dir' not in globals() or not Path(save_dir).exists():
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    save_dir = Path('/content/drive/MyDrive/hair_app/runs') / f'pixel3dmm_v4_preprocessing_{VID_NAME}_{run_id}'
    save_dir.mkdir(parents=True, exist_ok=False)
else:
    save_dir = Path(save_dir)

manifest = json.loads((root/'crop_meta'/'manifest.json').read_text())
for name in ['rgb','cropped','crop_meta','PIPnet_landmarks','PIPnet_annotated_images','pipnet','seg_og','seg_non_crop_annotations']:
    source = root / name
    if source.exists(): shutil.copytree(source, save_dir/name, dirs_exist_ok=True)

raw_dir = save_dir/'raw_inputs'
raw_dir.mkdir(exist_ok=True)
for item in manifest['items']:
    source = Path(INPUT_PATH)/item['source_name']
    assert source.exists(), source
    shutil.copy2(source, raw_dir/source.name)

landmark_dir = save_dir/'landmarks_json'
landmark_dir.mkdir(exist_ok=True)
all_landmarks, csv_rows = [], []
for item in manifest['items']:
    stem = Path(item['derived_name']).stem
    points = np.asarray(np.load(root/'PIPnet_landmarks'/f'{stem}.npy'), dtype=np.float64)
    assert points.shape == (98,2), (stem, points.shape)
    exported = []
    for index, (x, y) in enumerate(points):
        point = {
            'index': index, 'x_normalized': float(x), 'y_normalized': float(y),
            'x_crop_pixel': float(x*512.0), 'y_crop_pixel': float(y*512.0),
        }
        exported.append(point)
        csv_rows.append({
            'source_name': item['source_name'], 'frame': stem, **point,
        })
    payload = {
        'source_name': item['source_name'], 'frame': stem,
        'topology': 'PIPNet WFLW 98; indices 0-97',
        'coordinate_system': 'normalized final 512 crop; origin top-left; x right; y down',
        'points': exported,
    }
    (landmark_dir/f'{stem}.json').write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    all_landmarks.append(payload)
(save_dir/'landmarks_all.json').write_text(json.dumps(all_landmarks, indent=2, ensure_ascii=False))
with open(save_dir/'landmarks_long.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['source_name','frame','index','x_normalized','y_normalized','x_crop_pixel','y_crop_pixel'])
    writer.writeheader(); writer.writerows(csv_rows)

farl_labels = {
    0:'background',1:'neck',2:'face',3:'cloth',4:'right_ear_region',5:'left_ear_region',
    6:'right_brow',7:'left_brow',8:'right_eye',9:'left_eye',10:'nose',11:'inner_mouth',
    12:'lower_lip',13:'upper_lip',14:'hair',15:'eyeglasses',16:'hat_or_misc',
    17:'earring',18:'necklace_or_neck_detail',20:'uncovered_or_zero_logits',
}
segmentation_stats = []
for item in manifest['items']:
    stem = Path(item['derived_name']).stem
    mask = np.asarray(Image.open(root/'seg_og'/f'{stem}.png'))
    if mask.ndim == 3: mask = mask[...,0]
    values, counts = np.unique(mask, return_counts=True)
    total = int(mask.size)
    segmentation_stats.append({
        'source_name': item['source_name'], 'frame': stem, 'size': [int(mask.shape[1]),int(mask.shape[0])],
        'classes': [
            {'id':int(v),'name':farl_labels.get(int(v),'unknown'),'pixels':int(c),'ratio':float(c/total)}
            for v,c in zip(values,counts)
        ],
    })
(save_dir/'segmentation_statistics.json').write_text(json.dumps(segmentation_stats, indent=2, ensure_ascii=False))

if 'fig' in globals(): fig.savefig(save_dir/'preprocessing_overview.png', dpi=160, bbox_inches='tight')
logs_dir = save_dir/'logs'; logs_dir.mkdir(exist_ok=True)
for log in Path('/content').glob('p3dmm*.log'): shutil.copy2(log, logs_dir/log.name)
commit = subprocess.run(['git','-C','/content/pixel3dmm','rev-parse','HEAD'], capture_output=True, text=True).stdout.strip()
readme = '''Hair App Pixel3DMM V4 preprocessing bundle
raw_inputs/: immutable copies used for this run
cropped/: final 512 no-roll crops
crop_meta/: FaceBoxes candidates, chosen bbox, warnings, source<->crop matrices
PIPnet_landmarks/: original WFLW 98 normalized NPY files
landmarks_json/, landmarks_all.json, landmarks_long.csv: readable 98-point exports
PIPnet_annotated_images/: landmark overlays
seg_og/: raw FaRL label IDs
seg_non_crop_annotations/: colored FaRL previews
segmentation_statistics.json: per-frame label counts and ratios
preprocessing_overview.png: crop / landmarks / segmentation visual gate
artifact_sha256.json: integrity hashes
PRIVATE BIOMETRIC DATA: do not commit or publish.
'''
(save_dir/'README_PRIVATE.txt').write_text(readme)

hashes = {}
for path in sorted(p for p in save_dir.rglob('*') if p.is_file()):
    rel = str(path.relative_to(save_dir))
    hashes[rel] = {'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
(save_dir/'artifact_sha256.json').write_text(json.dumps(hashes, indent=2))
bundle = {
    'schema_version':'0.4', 'pipeline':'pixel3dmm_colab_v4', 'stage':'preprocessing',
    'pixel3dmm_commit':commit, 'input_set_id':VID_NAME, 'input_count':len(manifest['items']),
    'crop_contract':{'detector':'official FaceBoxesV2','selection':'confidence-first','margin':1.42,'size':[512,512],'roll_applied':False},
    'landmarks':{'model':'PIPNet WFLW 98','coordinates':'normalized final crop plus derived 512 pixel coordinates'},
    'segmentation':{'model':'FaRL celebM 448','raw_labels_preserved':True},
    'preprocessing_approved':bool(PREPROCESSING_APPROVED),
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'privacy':'private biometric artifacts; do not commit to git',
}
(save_dir/'preprocessing_bundle_manifest.json').write_text(json.dumps(bundle, indent=2, ensure_ascii=False))
print('COMPLETE PREPROCESSING BUNDLE SAVED:', save_dir)
print('landmark rows:', len(csv_rows), '| expected:', len(manifest['items'])*98)
print('hashed files:', len(hashes))


## 8. Pixel3DMM network inference 및 출력 개수 검증

아래 승인 셀은 crop·PIPNet 98점·FaRL mask를 사람이 확인하기 전 normal/UV와 tracking이 진행되는 것을 막는다. 공식 inference가 frame 내부 exception을 출력하고 계속 진행하므로 결과 개수도 별도로 검사한다.


In [ ]:
assert PREPROCESSING_APPROVED, (
    '먼저 위 crop/PIPNet 98/FaRL 결과를 확인하세요. '
    '문제가 없으면 위 셀의 PREPROCESSING_APPROVED를 True로 바꾸고 그 셀부터 다시 실행하세요.'
)
print('PREPROCESSING VISUAL APPROVAL: PASS')


In [ ]:
%%bash -s "$VID_NAME"
set -euo pipefail
cd /content/pixel3dmm
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=normals video_name="$1" 2>&1 | tee /content/p3dmm_normals.log
conda run -n p3dmm python scripts/network_inference.py model.prediction_type=uv_map video_name="$1" 2>&1 | tee /content/p3dmm_uv.log

In [ ]:
from pathlib import Path
root = Path('/content/p3dmm_preprocessed') / VID_NAME
expected = len(list((root / 'cropped').glob('*')))
normal_files = list((root / 'p3dmm' / 'normals').glob('*.png'))
uv_files = list((root / 'p3dmm' / 'uv_map').glob('*.png'))
print('expected/normals/uv:', expected, len(normal_files), len(uv_files))
assert len(normal_files) == expected, 'normal inference 일부/전체 실패: /content/p3dmm_normals.log 확인'
assert len(uv_files) == expected, 'UV inference 일부/전체 실패: /content/p3dmm_uv.log 확인'
print('network inference completeness: PASS')

## 9. Multi-image FLAME tracking

공식 README의 `iters=100 iters=1500` 중복은 config 설명에 맞춰 `iters=100 global_iters=1500`으로 교정했다. 입력 수보다 큰 default batch 16 때문에 crash하지 않도록 batch를 동적으로 낮춘다.

In [ ]:
%%bash -s "$VID_NAME"
set -euo pipefail
FLAME=/content/pixel3dmm/src/pixel3dmm/preprocessing/MICA/data
test -s "$FLAME/FLAME2020/generic_model.pkl"
test -s "$FLAME/FLAME2020/landmark_embedding.npy"
test -s "$FLAME/FLAME2020/FLAME_masks/FLAME_masks.pkl"
test -s "$FLAME/FLAME2023/flame2023_no_jaw.pkl"
N=$(find "/content/p3dmm_preprocessed/$1/cropped" -maxdepth 1 -type f | wc -l)
test "$N" -ge 2
BATCH=$N
if [ "$BATCH" -gt 16 ]; then BATCH=16; fi
echo "views=$N batch_size=$BATCH"
cd /content/pixel3dmm
conda run -n p3dmm python scripts/track.py video_name="$1" \
  iters=100 global_iters=1500 batch_size=$BATCH \
  include_neck=False w_exp=0.1 use_mouth_lmk=False \
  w_shape=0.01 w_shape_general=0.001 normal_super=2000.0 sil_super=1000.0 \
  use_flame2023=True ignore_mica=True is_discontinuous=True \
  2>&1 | tee /content/p3dmm_tracking.log

## 10. 3D mesh 확인

In [ ]:
!pip -q install trimesh plotly
import glob, trimesh
import plotly.graph_objects as go
cands = []
for ext in ('ply', 'obj'):
    cands += glob.glob(f'/content/p3dmm_tracking/**/*.{ext}', recursive=True)
cands = sorted(cands)
print('meshes:', len(cands), *cands[-10:], sep='\n')
assert cands, 'mesh 없음: /content/p3dmm_tracking.log의 첫 traceback 확인'
mesh_path = cands[-1]
mesh = trimesh.load(mesh_path, force='mesh')
assert len(mesh.vertices) and len(mesh.faces), 'mesh가 비어있음'
v, f = mesh.vertices, mesh.faces
fig = go.Figure(data=[go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2], color='lightgray', flatshading=True)])
fig.update_layout(scene=dict(aspectmode='data'), margin=dict(l=0,r=0,t=0,b=0))
fig.show()
print('mesh preview PASS:', mesh_path, '| vertices:', len(v), '| faces:', len(f))

## 11. 결과, raw logs, manifest를 Drive에 저장

성공·실패 여부와 관계없이 이 셀을 실행하면 다음 디버깅에 필요한 log를 보존한다.

In [ ]:
import datetime, glob, json, pathlib, shutil, subprocess, torch
base = pathlib.Path('/content/drive/MyDrive/hair_app')
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = base/'runs'/f'pixel3dmm_v4_{VID_NAME}_{run_id}'
for folder in ('logs','meshes','preprocessing'): (run_dir/folder).mkdir(parents=True,exist_ok=True)
for log in glob.glob('/content/p3dmm_*.log'): shutil.copy2(log,run_dir/'logs'/pathlib.Path(log).name)
mesh_files = []
for ext in ('ply','obj'): mesh_files += glob.glob(f'/content/p3dmm_tracking/**/*.{ext}',recursive=True)
for mesh in mesh_files: shutil.copy2(mesh,run_dir/'meshes'/pathlib.Path(mesh).name)
crop_manifest = root/'crop_meta'/'manifest.json'
if crop_manifest.exists(): shutil.copy2(crop_manifest,run_dir/'preprocessing'/crop_manifest.name)
commit = subprocess.run(['git','-C','/content/pixel3dmm','rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
manifest = {
    'pipeline':'pixel3dmm_colab_v4','preprocessing_contract_version':'0.4',
    'model':'pixel3dmm','commit':commit,'license':'CC BY-NC 4.0 (non-commercial)',
    'gpu':torch.cuda.get_device_name(0),'compute_capability':list(torch.cuda.get_device_capability(0)),
    'torch_cuda_arch_list':pathlib.Path('/content/p3dmm_torch_arch.txt').read_text().strip(),
    'input_set_id':VID_NAME,'input_count':len(imgs),
    'crop_config':'official FaceBoxes per-image bbox; margin=1.42; output=512; roll=false',
    'landmarks':'official PIPNet WFLW 98 on final crop',
    'tracking_config':'iters=100 global_iters=1500 dynamic_batch include_neck=False w_exp=0.1 use_mouth_lmk=False w_shape=0.01 w_shape_general=0.001 normal_super=2000 sil_super=1000 use_flame2023=True ignore_mica=True is_discontinuous=True',
    'fixes_applied':[
        'https clones','Cython before FaceBoxes','correct checkpoint paths','facer image_ids.long',
        'skip MICA preprocessing','zero MICA shape prior','validated FLAME2020+FLAME2023',
        'pinned DECA FLAME landmark embedding with SHA-256 verification','per-image FaceBoxes crop','no sparse-landmark roll',
        'PIPNet 98 and FaRL visual gate','dynamic batch','global_iters correction','output count validation',
    ],
    'mesh_count':len(mesh_files),'created_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
(run_dir/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
print('saved:',run_dir)
print(json.dumps(manifest,indent=2,ensure_ascii=False))


## 12. 평가

`scoring_sheet.csv`에 identity, geometry, hairline, side contour, scalp/ear topology, execution reliability를 1–5로 기록한다. Hidden scalp/rear는 측정값이 아니라 prior 추정이다.